In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [3]:
load_dotenv()

True

In [5]:
import os

groq_api_key = os.getenv("GROQ_API_KEY")

In [6]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key = groq_api_key,
    reasoning_format="hidden"

)

In [7]:
class SentimentSchema(BaseModel):
    sentiment: Literal['positve','negative'] = Field(description= "Sentiment of the reviews")

In [18]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')


In [19]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [10]:
prompt = "what is the sentiment of the following review - the software too good"
structured_model.invoke(prompt).sentiment

'positve'

In [12]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive','sentiment']
    diagnosis: dict
    response: str

In [20]:
def find_sentiment(state:ReviewState):
    prompt = f" for the following review find out the sentiment \n {state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    return {'sentiment':sentiment}

def check_sentiment(state:ReviewState)->Literal['positive_response','run_diagnosis']:
    if state['sentiment'] =='positive_response':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def positive_response(state:ReviewState):
    prompt = f"write a warm thankyou meassage in response tothis review :{state['review']} "
    response = model.invoke(prompt).content
    return{'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']
    
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}    


In [21]:
graph = StateGraph(ReviewState)
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [23]:
initial_state= {
    'review': 'the product was really good'
}

final_state = workflow.invoke(initial_state)


In [24]:
print(final_state)

{'review': 'the product was really good', 'sentiment': 'positve', 'diagnosis': {'issue_type': 'Other', 'tone': 'calm', 'urgency': 'low'}, 'response': 'Hello!\n\nThank you for reaching out. I understand you have an “Other” type of issue, and I’m here to help you sort it out. Since you mentioned everything is calm and the matter isn’t urgent, we can take a little time to make sure we get to the right solution together.\n\n### Here’s a quick way to move forward:\n\n1. **Give me a bit more detail**  \n   - What specifically are you experiencing? (e.g., a feature that isn’t behaving as expected, a question about account settings, a billing curiosity, etc.)  \n   - If there’s an error message, screen, or step that seems odd, let me know what it says or looks like.\n\n2. **Check the basics**  \n   - **Refresh / Re‑login:** Sometimes a simple page refresh or logging out and back in can clear minor hiccups.  \n   - **Clear cache/cookies:** If you’re using a web interface, clearing your browser’